# 10.03 -- Medusa: Multiple Decoding Heads for Parallel Token Prediction

**Module:** 10 -- Advanced Inference  
**Notebook:** 3 of 5  
**Prerequisites:** 10.01 (Speculative Decoding), 10.02 (EAGLE)

---

## Learning Objectives

1. Understand how Medusa replaces the draft model with multiple prediction heads
2. Implement Medusa heads from scratch and understand the parameter budget
3. Implement tree attention for verifying multiple candidate sequences simultaneously
4. Measure the effect of head count and tree width on throughput
5. Compare Medusa with EAGLE and understand when each method is preferable

---

## 1. The Medusa Approach

EAGLE and standard speculative decoding both require a separate drafting step before
the target model verifies. Medusa takes a different architectural approach: it adds
multiple prediction heads directly on top of the final transformer layer, each
predicting a token further into the future than the original LM head.

```
Standard LM head:
  hidden_state_t --> [LM head] --> P(token_t+1)

Medusa with 4 heads:
  hidden_state_t --> [LM head 0] --> P(token_t+1)   (standard)
  hidden_state_t --> [Medusa head 1] --> P(token_t+2)
  hidden_state_t --> [Medusa head 2] --> P(token_t+3)
  hidden_state_t --> [Medusa head 3] --> P(token_t+4)
  hidden_state_t --> [Medusa head 4] --> P(token_t+5)
```

All heads run in parallel on the same hidden state. One forward pass through the target
model produces candidate predictions for the next 5 tokens simultaneously.

The verification step uses **tree attention** to check multiple candidate sequences
efficiently in one forward pass, then accepts a prefix of the tree that the original
model agrees with.


In [ ]:
# Environment setup.
#
# Medusa is implemented as an extension to an existing model checkpoint.
# The Medusa heads are a small addendum (a few FF layers) added after the
# final transformer layer. We implement everything from scratch here using
# GPT-2 as the base model.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
import itertools
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


## 2. Medusa Head Architecture


In [ ]:
# Medusa head implementation.
#
# Each Medusa head is a two-layer feed-forward network with a residual connection.
# Architecture per head:
#   hidden_state --> Linear(hidden, hidden) --> SiLU --> Linear(hidden, vocab) --> logits
#   with a skip connection from input to output (residual).
#
# Why two layers instead of one? The first layer lets the head learn a specialised
# representation for multi-step lookahead; the second projects to vocabulary.
# A single linear layer would be equivalent to a rank-1 approximation of the
# lookahead distribution, which is too restrictive.
#
# Parameter count per head for GPT-2 (hidden=768, vocab=50257):
#   Layer 1: 768 * 768 = 589 K
#   Layer 2: 768 * 50257 = 38.6 M
#   Total per head: ~39 M
#   5 heads: ~195 M  on top of a 117 M target  (166 percent overhead)
#
# For a 7B target (hidden=4096, vocab=32000):
#   Per head: ~132 M
#   5 heads:  ~660 M  on top of 7000 M target  (9.4 percent overhead)
#   This is why Medusa is most cost-effective on large models.

class MedusaHead(nn.Module):
    """
    A single Medusa prediction head.

    Predicts token probabilities at a fixed lookahead offset (1-step ahead,
    2-steps ahead, etc.) from the same hidden state.
    """

    def __init__(self, hidden_size: int, vocab_size: int):
        super().__init__()
        # Residual feed-forward block
        self.linear1   = nn.Linear(hidden_size, hidden_size, bias=False)
        self.act       = nn.SiLU()
        self.lm_head   = nn.Linear(hidden_size, vocab_size, bias=False)

        # Initialise near zero so early training starts close to the base LM head
        nn.init.zeros_(self.linear1.weight)
        nn.init.normal_(self.lm_head.weight, std=0.02)

    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        """
        Args:
            hidden_state: (B, hidden_size) -- final layer hidden state
        Returns:
            logits: (B, vocab_size)
        """
        # Residual: add the transformed representation to the original
        residual = hidden_state
        x = self.act(self.linear1(hidden_state))
        x = residual + x        # residual connection keeps early training stable
        return self.lm_head(x)


class MedusaModel(nn.Module):
    """
    Wraps a pretrained language model with Medusa heads.

    The base model weights are frozen; only the Medusa heads are trained.
    This is Medusa-1 (the faster-to-train variant). Medusa-2 fine-tunes
    both the base model and the heads jointly but requires more compute.
    """

    def __init__(self, base_model, num_heads: int = 4, vocab_size: int = 50257):
        super().__init__()
        self.base_model = base_model
        self.num_heads  = num_heads

        hidden_size = base_model.config.n_embd   # GPT-2: 768

        # One head per lookahead offset: head k predicts token at t+k+2
        # (head 0 predicts t+2 since the base LM head already predicts t+1)
        self.medusa_heads = nn.ModuleList([
            MedusaHead(hidden_size, vocab_size)
            for _ in range(num_heads)
        ])

        # Freeze base model weights
        for p in self.base_model.parameters():
            p.requires_grad = False

    def forward(self, input_ids: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Run the base model and all Medusa heads.

        Returns:
            base_logits:   (B, seq, vocab) -- standard LM head output
            medusa_logits: list of num_heads tensors, each (B, seq, vocab)
        """
        with torch.no_grad():
            out = self.base_model(input_ids, output_hidden_states=True)

        base_logits   = out.logits                         # (B, seq, vocab)
        last_hidden   = out.hidden_states[-1]              # (B, seq, hidden)

        medusa_logits = [head(last_hidden) for head in self.medusa_heads]
        return base_logits, medusa_logits


from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer  = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained('gpt2', output_hidden_states=True).to(device)
base_model.eval()

NUM_HEADS    = 4
medusa_model = MedusaModel(base_model, num_heads=NUM_HEADS, vocab_size=50257).to(device)
medusa_model.eval()

base_params   = sum(p.numel() for p in base_model.parameters())
head_params   = sum(p.numel() for p in medusa_model.medusa_heads.parameters())
print(f'Base model parameters:    {base_params:>12,}')
print(f'Medusa heads ({NUM_HEADS} heads): {head_params:>12,}')
print(f'Total:                    {base_params + head_params:>12,}')
print(f'Head overhead:            {head_params / base_params:.1%}')


## 3. Candidate Generation and Tree Attention


In [ ]:
# Candidate generation: building the draft tree.
#
# Medusa generates candidates by taking the top-k tokens from each head.
# With 4 heads and top-k=2, we have 2^4 = 16 candidate sequences of length 5.
# With top-k=3, we have 3^4 = 81 candidates.
#
# In practice, not all candidates are evaluated: a budget B limits the total
# number of tree nodes. The candidates are ordered by product of their
# per-head probabilities and only the top B are kept.
#
# Tree attention: the candidates share a common prefix (the original prompt).
# Tree attention uses a custom attention mask that prevents candidates from
# attending to each other's future tokens, while allowing them to share
# computation over the common prefix.
#
# For simplicity, we implement flat (non-tree) verification here:
# we verify each candidate sequence independently. The tree attention
# optimisation is described in the Engineering Notes.

def medusa_generate_candidates(
    base_logits:   torch.Tensor,        # (vocab,) logits at last position
    medusa_logits: List[torch.Tensor],  # list of num_heads (vocab,) logits
    top_k:         int = 2,
) -> Tuple[List[List[int]], List[float]]:
    """
    Build candidate sequences from base + Medusa head predictions.

    Each candidate is a sequence [token_1, token_2, ..., token_{H+1}] where:
      token_1 is from the base LM head (next token)
      token_k is from Medusa head k-1 (lookahead token k)

    Returns candidates as lists of token ids and their log-probability scores.
    """
    # Top-k tokens from the base head
    base_probs    = F.softmax(base_logits, dim=-1)
    base_topk     = torch.topk(base_probs, top_k)
    base_tokens   = base_topk.indices.tolist()
    base_logprobs = base_topk.values.log().tolist()

    # Top-k tokens from each Medusa head
    head_tokens_list   = []
    head_logprobs_list = []
    for h_logits in medusa_logits:
        h_probs  = F.softmax(h_logits, dim=-1)
        h_topk   = torch.topk(h_probs, top_k)
        head_tokens_list.append(h_topk.indices.tolist())
        head_logprobs_list.append(h_topk.values.log().tolist())

    # Cartesian product: all combinations of top-k choices per head
    all_indices = [range(top_k)] * (1 + len(medusa_logits))
    candidates  = []
    scores      = []

    for combo in itertools.product(*all_indices):
        seq   = [base_tokens[combo[0]]]
        score = base_logprobs[combo[0]]
        for h, idx in enumerate(combo[1:]):
            seq.append(head_tokens_list[h][idx])
            score += head_logprobs_list[h][idx]
        candidates.append(seq)
        scores.append(score)

    # Sort by score descending
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    return [candidates[i] for i in order], [scores[i] for i in order]


def medusa_verify(
    base_model,
    prompt_ids:   torch.Tensor,   # (1, prompt_len)
    candidates:   List[List[int]],
    max_budget:   int = 8,
) -> Tuple[List[int], int]:
    """
    Verify candidate sequences against the base model.

    For each candidate, check how many tokens the base model would have
    generated identically (greedy). Accept the longest prefix that matches.

    Returns the best accepted sequence and its length.

    Note: this simplified implementation runs one forward pass per candidate.
    Production Medusa uses tree attention to verify all candidates in one pass.
    """
    best_seq    = [candidates[0][0]]   # at minimum accept first token
    best_len    = 1

    for cand in candidates[:max_budget]:
        full_ids = torch.cat([
            prompt_ids,
            torch.tensor(cand, device=device).unsqueeze(0)
        ], dim=1)

        with torch.no_grad():
            logits = base_model(full_ids).logits[0]   # (seq, vocab)

        # Check greedy match at each candidate position
        prompt_len = prompt_ids.shape[1]
        accepted   = []
        for i, tok in enumerate(cand):
            greedy = logits[prompt_len - 1 + i].argmax().item()
            if greedy == tok:
                accepted.append(tok)
            else:
                accepted.append(greedy)   # take the correct token instead
                break

        if len(accepted) > best_len:
            best_seq = accepted
            best_len = len(accepted)

    return best_seq, best_len


# Demo
prompt  = 'Transformers use self-attention to'
ids     = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

base_logits_all, medusa_logits_all = medusa_model(ids)

# Extract last-position logits
base_last   = base_logits_all[0, -1, :]          # (vocab,)
medusa_last = [m[0, -1, :] for m in medusa_logits_all]  # list of (vocab,)

candidates, scores = medusa_generate_candidates(base_last, medusa_last, top_k=2)

print(f'Prompt: "{prompt}"')
print(f'Generated {len(candidates)} candidates from {NUM_HEADS} heads (top-2 per head):')
print()
for i, (cand, score) in enumerate(zip(candidates[:5], scores[:5])):
    tokens_text = tokenizer.decode(cand)
    print(f'  Candidate {i+1} (logprob={score:.2f}): [{tokens_text}]')
print(f'  ... ({len(candidates)} total)')
print()

best, best_len = medusa_verify(base_model, ids, candidates, max_budget=8)
print(f'Verified. Best accepted sequence ({best_len} tokens): [{tokenizer.decode(best)}]')


In [ ]:
# Medusa speedup experiment: heads, top-k, and budget.
#
# The speedup from Medusa depends on three hyperparameters:
#
#   1. num_heads (H): how many lookahead positions to predict.
#      More heads = more candidates = higher potential speedup.
#      But each head adds to the parameter count and memory.
#
#   2. top_k (K): how many candidate tokens to keep per head.
#      Higher K exponentially increases candidates (K^H).
#      In practice K=2 or K=3 is used with a budget cap.
#
#   3. budget (B): maximum candidates to verify per forward pass.
#      Higher budget improves acceptance but requires more compute.
#      The optimal B balances verification cost vs acceptance gain.
#
# For random Medusa heads (no training), acceptance is essentially random.
# After training, acceptance rates reach 75-85 percent at depth 1,
# falling off at greater depths. The effective speedup formula is:
#   speedup ~ sum_{d=1}^{H} product_{i=1}^{d} alpha_i
# where alpha_i is the acceptance rate at depth i.

TEST_PROMPTS = [
    'The transformer model was introduced',
    'Gradient descent is an optimisation algorithm',
    'The key idea behind attention is',
    'Language models are trained on large',
    'The BERT model uses bidirectional',
    'Neural networks learn by adjusting',
    'The ReLU activation function outputs',
    'Tokenisation splits text into',
]

# Measure mean accepted tokens per Medusa call across prompts
def measure_medusa_acceptance(num_heads, top_k, budget, prompts):
    """Return mean accepted sequence length over a list of prompts."""
    model = MedusaModel(base_model, num_heads=num_heads, vocab_size=50257).to(device)
    model.eval()
    accepted_lens = []

    for prompt in prompts:
        ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
        bl_all, ml_all = model(ids)
        bl   = bl_all[0, -1, :]
        ml   = [m[0, -1, :] for m in ml_all]
        cands, _ = medusa_generate_candidates(bl, ml, top_k=top_k)
        _, best_len = medusa_verify(base_model, ids, cands, max_budget=budget)
        accepted_lens.append(best_len)

    return np.mean(accepted_lens)


print('Measuring Medusa acceptance across configurations...')
print('(Using random head weights -- trained weights would give higher acceptance)')
print()

# Sweep num_heads
head_results = {}
for h in [1, 2, 3, 4]:
    mean_acc = measure_medusa_acceptance(h, top_k=2, budget=8, prompts=TEST_PROMPTS)
    head_results[h] = mean_acc
    print(f'  num_heads={h}: mean accepted tokens = {mean_acc:.2f}')

print()

# Sweep top_k
topk_results = {}
for k in [1, 2, 3]:
    mean_acc = measure_medusa_acceptance(3, top_k=k, budget=16, prompts=TEST_PROMPTS)
    topk_results[k] = mean_acc
    print(f'  top_k={k}: mean accepted tokens = {mean_acc:.2f}')


In [ ]:
# Visualise Medusa results and comparison with other methods.
#
# Four panels:
#   1. Mean accepted tokens vs number of Medusa heads (sweep above).
#   2. Candidate tree diagram: visualise how top-2 per head generates a tree.
#   3. Acceptance rate by lookahead depth (empirical: acceptance falls with depth).
#   4. Method comparison: autoregressive / speculative / EAGLE / Medusa.

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.32)

# Panel 1: tokens accepted vs num_heads
ax = fig.add_subplot(gs[0, 0])
hs = list(head_results.keys())
vs = list(head_results.values())
ax.bar(hs, vs, color='#2ca02c', alpha=0.85, width=0.5)
for h, v in zip(hs, vs):
    ax.text(h, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)
ax.set_xlabel('Number of Medusa heads', fontsize=11)
ax.set_ylabel('Mean accepted tokens per forward pass', fontsize=11)
ax.set_title('Accepted Tokens vs Head Count\n(Random weights; trained gives 2-4x higher)', fontsize=12)
ax.grid(alpha=0.3, axis='y')

# Panel 2: candidate tree visualisation
ax = fig.add_subplot(gs[0, 1])
ax.axis('off')

# Draw a simple tree for 2 heads, top-2
# Root -> 2 base tokens -> 2 medusa-1 tokens each -> 2 medusa-2 tokens each
root = (0.5, 0.95)

def draw_tree_node(ax, x, y, label, color='#1f77b4'):
    ax.add_patch(plt.Circle((x, y), 0.04, color=color, zorder=5))
    ax.text(x, y - 0.08, label, ha='center', va='top', fontsize=8)

def draw_edge(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2 + 0.04), xytext=(x1, y1 - 0.04),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))

# Level 0: root (prompt)
ax.add_patch(plt.Circle(root, 0.04, color='#ff7f0e', zorder=5))
ax.text(root[0], root[1] + 0.06, 'hidden\nstate', ha='center', fontsize=8)

# Level 1: base LM head, top-2
level1 = [(0.25, 0.70), (0.75, 0.70)]
labels1 = ['A (p=0.41)', 'B (p=0.28)']
for (x, y), lbl in zip(level1, labels1):
    draw_tree_node(ax, x, y, lbl, '#2ca02c')
    draw_edge(ax, *root, x, y)

# Level 2: Medusa head 1, top-2 for each
level2_left  = [(0.12, 0.45), (0.38, 0.45)]
level2_right = [(0.62, 0.45), (0.88, 0.45)]
labels2 = [['C', 'D'], ['E', 'F']]
for parent, children, labs in zip(level1, [level2_left, level2_right], labels2):
    for (cx, cy), lbl in zip(children, labs):
        draw_tree_node(ax, cx, cy, lbl, '#d62728')
        draw_edge(ax, *parent, cx, cy)

ax.set_xlim(0, 1)
ax.set_ylim(0.2, 1.05)
ax.set_title('Medusa Candidate Tree\n(2 heads, top-2 per head = 4 leaf candidates)', fontsize=12)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#ff7f0e', label='Prompt hidden state'),
    Patch(color='#2ca02c', label='Base LM head (depth 1)'),
    Patch(color='#d62728', label='Medusa head 1 (depth 2)'),
], loc='lower right', fontsize=8)

# Panel 3: acceptance by depth (representative trained model values)
ax = fig.add_subplot(gs[1, 0])
depths = [1, 2, 3, 4, 5]
# Trained acceptance rates decay with depth (from Medusa paper, LLaMA-2 7B)
acceptance_trained = [0.82, 0.68, 0.54, 0.42, 0.31]
# Random weights: essentially uniform guess
acceptance_random  = [0.12, 0.10, 0.09, 0.08, 0.07]
ax.plot(depths, acceptance_trained, 'o-', color='#2ca02c', lw=2, ms=7,
        label='Trained Medusa (LLaMA-2 7B, from paper)')
ax.plot(depths, acceptance_random,  's--', color='#d62728', lw=2, ms=7,
        label='Random heads (no training)')
cum_acc = np.cumprod(acceptance_trained)
ax.fill_between(depths, 0, cum_acc, alpha=0.12, color='#2ca02c',
                label='Cumulative acceptance (area = speedup)')
ax.set_xlabel('Lookahead depth', fontsize=11)
ax.set_ylabel('Acceptance rate at this depth', fontsize=11)
ax.set_title('Acceptance Rate vs Lookahead Depth\n(Trained Medusa, representative values from paper)', fontsize=12)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3)

# Panel 4: method comparison
ax = fig.add_subplot(gs[1, 1])
methods   = ['Autoregressive', 'Spec. decoding\n(small model)', 'Medusa\n(trained)',
             'EAGLE\n(trained)', 'EAGLE-2\n(dynamic)']
speedups  = [1.0, 2.1, 2.5, 3.0, 3.4]
overheads = [0, 500, 200, 150, 150]   # MB of extra weight
bar_colors = ['#aec7e8', '#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
bars = ax.bar(methods, speedups, color=bar_colors, alpha=0.85, width=0.55)
for b, v in zip(bars, speedups):
    ax.text(b.get_x() + b.get_width()/2, v + 0.04, f'{v}x',
            ha='center', fontsize=10, fontweight='bold')
ax_r = ax.twinx()
ax_r.plot(methods, overheads, 'D--', color='purple', lw=1.5, ms=7, label='Weight overhead (MB)')
ax_r.set_ylabel('Extra weight overhead (MB)', color='purple', fontsize=10)
ax_r.tick_params(axis='y', labelcolor='purple')
ax.set_ylabel('Throughput speedup', fontsize=11)
ax.set_title('Method Comparison: Speedup vs Overhead\n(Representative values, 7B model)', fontsize=12)
ax.set_ylim(0, 4.2)
ax.grid(alpha=0.3, axis='y')

plt.suptitle('Medusa Decoding Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/10_medusa.png', dpi=150, bbox_inches='tight')
plt.show()

eff_speedup = 1 + sum(np.cumprod(acceptance_trained))
print(f'Effective speedup from depth-acceptance curve: {eff_speedup:.2f}x')
print('(1 token from autoregressive + sum of cumulative acceptance per depth)')


## 4. Tree Attention

The simplified verification above runs one target model forward pass per candidate.
Production Medusa uses tree attention to verify all candidates in a single forward pass.

```
Standard attention mask (causal, 4-token sequence):
  Token  1  2  3  4
    1    1  0  0  0
    2    1  1  0  0
    3    1  1  1  0
    4    1  1  1  1

Tree attention mask (2 candidates sharing prefix [1,2]):
  Cand A: [1, 2, 3a, 4a]
  Cand B: [1, 2, 3b, 4b]

  Token  1  2  3a  4a  3b  4b
    1    1  0   0   0   0   0
    2    1  1   0   0   0   0
   3a    1  1   1   0   0   0   <- can see shared prefix but not branch B
   4a    1  1   1   1   0   0
   3b    1  1   0   0   1   0   <- can see shared prefix but not branch A
   4b    1  1   0   0   1   1
```

Tree attention verifies all branches simultaneously without contaminating
one branch with the other's speculative tokens. This is the key computational
optimisation that makes Medusa practical for large trees.

## 5. Engineering Notes

**Medusa-1 vs Medusa-2**

| Variant | Training | Acceptance rate | Use case |
|---------|----------|-----------------|----------|
| Medusa-1 | Freeze base model, train heads only (~1 GPU-hour) | 75-82% at depth 1 | Fast to deploy, minimal risk |
| Medusa-2 | Joint fine-tune base + heads (~10-100 GPU-hours) | 80-88% at depth 1 | Maximum quality |

**When to prefer Medusa over EAGLE**

| Situation | Recommendation |
|-----------|---------------|
| Cannot access target model hidden states | Use Medusa (heads operate on final layer output only) |
| Need plug-and-play with any HF model | Medusa is easier to attach |
| Maximum acceptance rate is priority | EAGLE (feature-level drafting is more accurate) |
| Very large models (70B+) | Both work; EAGLE has lower relative overhead |

## 6. Exercises

1. **Tree attention mask**: Implement the tree attention mask for a Medusa tree with 2 heads and top-2 per head (4 leaves). Verify that the mask correctly prevents cross-contamination between branches by checking that masking out branch B tokens does not change branch A verification results.

2. **Medusa training step**: Implement one gradient step for Medusa-1. Given a batch of (prompt, continuation) pairs, compute cross-entropy loss for each head against the true future tokens (head k targets token at position t+k+1). Plot per-head loss curves.

3. **Budget ablation**: Measure mean accepted tokens for budget values [1, 2, 4, 8, 16] with 4 heads and top-2. Plot accepted tokens vs budget. At what budget does the curve flatten (diminishing returns)?

4. **Medusa vs greedy**: Compare output quality between standard greedy decoding and Medusa greedy decoding (accepting only top-1 per head, no stochastic acceptance). They should produce identical outputs if the heads are accurate. Measure the fraction of mismatches over 20 prompts.

5. **Dynamic head selection**: Implement an adaptive Medusa that disables heads when their entropy is high (predicting a uniform distribution). This is the Medusa analogue of EAGLE-2 dynamic gamma. Measure mean accepted tokens before and after adaptation.

---

## 7. References

- [Cai et al. (2024) -- Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads](https://arxiv.org/abs/2401.10774)
- [Li et al. (2024) -- EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty](https://arxiv.org/abs/2401.15077)
- [Miao et al. (2024) -- SpecInfer: Accelerating LLM Serving with Tree-based Speculative Inference and Verification](https://arxiv.org/abs/2305.09781)
